# SplitSeek-Pro Model Training Notebook

This notebook is based on the training.py script and demonstrates the workflow for training a SplitSeek-Pro model, including data preparation, model construction, training and validation, and model saving.

## 0. Install requirements
Install all required Python packages for this project using pip.

In [ ]:
!pip install -r ../requirements.txt

## 1. Import Required Libraries
Import PyTorch, NumPy, os, json, time, and other dependencies, as well as custom utils and model_utils modules.

In [ ]:
import os
import sys
import time
import json
import numpy as np
import torch
from torch.utils.data import DataLoader
from utils import worker_init_fn, get_pdbs, loader_pdb, build_training_clusters, get_std_opt, PDBDataset, StructureLoader, StructureDataset
from model_utils import featurize, loss_mse, SplitSeek
import warnings
import random

## 2. Parse Arguments
Please download the fine-tuning dataset from https://zenodo.org/records/17421132, extract the 'cp_pdb' and 'cp_pdb_esm' folders and move them into the 'data/' directory of this project. Update **path_for_training_data** with the path to your dataset. The target folder must include the required split files (train/valid/test.txt) and feature directories (cp_pdb, cp_pdb_esm). Other parameters are pre-set but can be modified.

In [ ]:
import types
args = types.SimpleNamespace(
    path_for_training_data="./data",
    path_for_outputs="./finetune_outputs",
    previous_checkpoint="../weights/pretrain_weights.pt",
    num_epochs=50,
    save_model_every_n_epochs=2,
    reload_data_every_n_epochs=10,
    num_examples_per_epoch=1000000,
    batch_size=10000,
    max_protein_length=10000,
    hidden_dim=128,
    num_encoder_layers=3,
    num_decoder_layers=3,
    num_neighbors=48,
    dropout=0.1,
    backbone_noise=0.1,
    rescut=3.5,
    debug=False,
    gradient_norm=-1.0,
    mixed_precision=True
)
args

## 3. Prepare Training and Validation Datasets
Load training and validation files, build PDBDataset and DataLoader, and process aaindex.json and other auxiliary data.

In [ ]:
data_path = args.path_for_training_data

with open(f'{data_path}/aaindex.json', 'r') as f:
    aaindex1 = json.load(f)

params = {
    "TRAIN"     : f"{data_path}/train.txt",
    "VAL"       : f"{data_path}/valid.txt",
    "TEST"      : f"{data_path}/test.txt",
    "DIR"       : f"{data_path}",
}

LOAD_PARAM = {
    'batch_size' : 1,
    'shuffle'    : True,
    'pin_memory' : False,
    'num_workers': 4
}

if args.debug:
    args.num_examples_per_epoch = 50
    args.max_protein_length = 1000
    args.batch_size = 1000

train, valid, test = build_training_clusters(params, args.debug)

train_set = PDBDataset(train, loader_pdb, params)
train_loader = DataLoader(train_set, worker_init_fn=worker_init_fn, **LOAD_PARAM)
valid_set = PDBDataset(valid, loader_pdb, params)
valid_loader = DataLoader(valid_set, worker_init_fn=worker_init_fn, **LOAD_PARAM)

## 4. Build Model and Optimizer
Initialize the SplitSeek model, set device (GPU/CPU), load pretrained weights (if any), and build the optimizer.

In [ ]:
import torch.multiprocessing

torch.multiprocessing.set_sharing_strategy('file_system')

scaler = torch.amp.GradScaler('cuda')
device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")

model = SplitSeek(
    aaindex1=aaindex1,
    node_features=args.hidden_dim,
    edge_features=args.hidden_dim,
    hidden_dim=args.hidden_dim,
    num_encoder_layers=args.num_encoder_layers,
    num_decoder_layers=args.num_decoder_layers,
    k_neighbors=args.num_neighbors,
    dropout=args.dropout,
    augment_eps=args.backbone_noise
)
model.to(device)

PATH = args.previous_checkpoint
if PATH:
    checkpoint = torch.load(PATH)
    model.load_state_dict(checkpoint['model_state_dict'])

total_step = 0
epoch = 0

optimizer = get_std_opt(model.parameters(), args.hidden_dim, total_step)
min_valid_loss = 10
if PATH:
    optimizer.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

## 5. Training and Validation Loop
Implement the main training loop, including forward pass, loss calculation, backward pass, gradient clipping, mixed precision training, and validation evaluation.

In [ ]:
pdb_dict_train = get_pdbs(train_loader, 1, args.max_protein_length, args.num_examples_per_epoch)
pdb_dict_valid = get_pdbs(valid_loader, 1, args.max_protein_length, args.num_examples_per_epoch)

dataset_train = StructureDataset(pdb_dict_train, truncate=None, max_length=args.max_protein_length)
dataset_valid = StructureDataset(pdb_dict_valid, truncate=None, max_length=args.max_protein_length)

loader_train = StructureLoader(dataset_train, batch_size=args.batch_size)
loader_valid = StructureLoader(dataset_valid, batch_size=args.batch_size)

logfile = os.path.join(time.strftime(args.path_for_outputs, time.localtime()), 'log.txt')
if not PATH:
    os.makedirs(os.path.dirname(logfile), exist_ok=True)
    with open(logfile, 'w') as f:
        f.write('Epoch\tTrain\tValidation\n')

for e0 in range(args.num_epochs):
    t0 = time.time()
    e = epoch + e0
    model.train()
    train_sum, train_weight = 0., 0.

    for k, batch in enumerate(loader_train):
        X, S, ESM_S, L, mask, lengths, residue_idx, mask_self, chain_encoding_all, batch_name = featurize(batch, device)
        optimizer.zero_grad()
        mask_for_loss = mask*mask_self

        if args.mixed_precision:
            with torch.amp.autocast('cuda'):
                probs = model(X, S, ESM_S, mask, residue_idx, chain_encoding_all)
                loss, loss_av = loss_mse(L ,probs, mask_for_loss)
            scaler.scale(loss_av).backward()
            if args.gradient_norm > 0.0:
                total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.gradient_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            probs = model(X, S, mask, residue_idx, chain_encoding_all)
            loss, loss_av = loss_mse(L ,probs, mask)
            if args.gradient_norm > 0.0:
                total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.gradient_norm)
            optimizer.step()

        train_sum += torch.sum(loss * mask_for_loss).cpu().data.numpy()
        train_weight += torch.sum(mask_for_loss).cpu().data.numpy()
        total_step += 1

    model.eval()
    with torch.no_grad():
        validation_sum, validation_weights = 0., 0.
        for _, batch in enumerate(loader_valid):
            X, S, ESM_S, L, mask, lengths, residue_idx, mask_self, chain_encoding_all, batch_name = featurize(batch, device)
            probs = model(X, S, ESM_S, mask, residue_idx, chain_encoding_all)
            mask_for_loss = mask*mask_self
            loss, loss_av = loss_mse(L, probs, mask)
            validation_sum += torch.sum(loss * mask_for_loss).cpu().data.numpy()
            validation_weights += torch.sum(mask_for_loss).cpu().data.numpy()

    train_loss = train_sum / (train_weight + 1e-9)
    train_perplexity = np.exp(1 + train_loss)
    validation_loss =  validation_sum / (validation_weights + 1e-9)
    validation_perplexity = np.exp(1 + validation_loss)

    train_perplexity_ = np.format_float_positional(np.float32(train_perplexity), unique=False, precision=3)
    validation_perplexity_ = np.format_float_positional(np.float32(validation_perplexity), unique=False, precision=3)

    t1 = time.time()
    dt = np.format_float_positional(np.float32(t1 - t0), unique=False, precision=1)
    with open(logfile, 'a') as f:
        f.write(f'epoch: {e+1}, step: {total_step}, time: {dt}, train: {train_perplexity_}, valid: {validation_perplexity_}\n')

    print(f'epoch: {e+1}, step: {total_step}, time: {dt}, train: {train_perplexity_}, valid: {validation_perplexity_}')
    checkpoint_filename_last = os.path.join(os.path.dirname(logfile), 'model_weights/epoch_last.pt')
    best_checkpoint_filename = os.path.join(os.path.dirname(logfile), 'model_weights/best_weights.pt')
    torch.save({
        'epoch': e+1,
        'step': total_step,
        'num_edges' : args.num_neighbors,
        'noise_level': args.backbone_noise,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.optimizer.state_dict(),
    }, checkpoint_filename_last)

    if validation_perplexity <= min_valid_loss:
        torch.save({
            'epoch': e+1,
            'step': total_step,
            'num_edges' : args.num_neighbors,
            'noise_level': args.backbone_noise,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.optimizer.state_dict(),
        }, best_checkpoint_filename)
        min_valid_loss = validation_perplexity

    if (e+1) % args.save_model_every_n_epochs == 0:
        checkpoint_filename = os.path.join(os.path.dirname(logfile), f'model_weights/epoch{e+1}_step{total_step}.pt')
        torch.save({
            'epoch': e+1,
            'step': total_step,
            'num_edges' : args.num_neighbors,
            'noise_level': args.backbone_noise, 
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.optimizer.state_dict(),
        }, checkpoint_filename)